In [1]:
import json
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from collections import defaultdict

import math
from scipy.stats import t


In [2]:
# define constants
beta = 10.0  

tmin = 1
tmax = 10

unit = 1000

# Generate time periods
T = [*range(tmin, tmax +1)]

# Calculate delta values
delta = {t: (1 + 0.03) ** t for t in T}

file_path = 'min_unvax_base.json'
# Load the JSON file
with open(file_path, 'r') as file:
    data = json.load(file)

    #scenario probabilities
with open('scenario_pair_probabilities_new.json', 'r') as f:
    probabilities = json.load(f)


In [3]:
from collections import defaultdict

def process_scenarios(S_data, beta, delta):
    # Step 1: Reorganize data with scenario at the top level
    S_data_by_scenario = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    for antigen, year_data in S_data.items():
        for year, scenario_data in {y: d for y, d in year_data.items() if y != '0'}.items():
            for scenario, value in scenario_data.items():
                S_data_by_scenario[scenario][year][antigen] = value

    # Convert to regular dictionary
    S_data_by_scenario = dict(S_data_by_scenario)

    # Step 2: Scale data
    S_data_by_scenario_scaled = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    for scenario, years in S_data_by_scenario.items():
        for year, antigens in years.items():
            for antigen in antigens.keys():
                S_data_by_scenario_scaled[scenario][year][antigen] = (
                    S_data_by_scenario[scenario][year][antigen] * beta / delta[int(year)]
                )

    # Step 3: Calculate scenario sums
    scenario_sums = {}
    for scenario, years in S_data_by_scenario_scaled.items():
        scenario_sum = sum(
            value
            for year in years.values()
            for value in year.values()
            if isinstance(value, (int, float))
        )
        scenario_sums[scenario] = scenario_sum

    return scenario_sums


In [4]:
S_data = data['S']
scenario_sums = process_scenarios(S_data, beta, delta)

In [5]:
S_OBJ_Values = {k: probabilities[k] * scenario_sums[k] for k in probabilities}
S_OBJ_Value = sum(S_OBJ_Values.values())
# S_OBJ_Value
print(f"Missed Dose OBJ Value: {(S_OBJ_Value)}")

Missed Dose OBJ Value: 152122942.46344373


G(x) Function - OBJ function gap estimate

In [17]:
def G_n(x_hat, xi_samples):
    """
    Computes the function G_n(x_hat) as defined.

    Parameters:
    x_hat (vector): A vector of values, that represents the missed doses by scenario
    xi_samples (vector): A vector of values, that represent the second solution missed doses by scenario

    Returns:
    float: The computed value of G_n(x_hat).
    """
    n = len(scenario_sums)
    term_1 = mean_value = sum(x_hat.values()) / len(x_hat)
    term_2 = mean_value = sum(xi_samples.values()) / len(xi_samples)
    return term_1 - term_2


variance function - using x_hat and new x_star

In [28]:
def calculate_sample_variance(x_hat, x_star, f_hat_n_x, f_hat_n_x_star):
    """
    Calculate the sample variance s_n^2(x_star_n).

    Parameters:
    x_hat (list or array): Values for f(x_hat, xi).
    x_star (list or array): Values for f(x_star, xi).
    f_hat_n_x (float): Sample mean of f(x_hat, xi).
    f_hat_n_x_star (float): Sample mean of f(x_star, xi).

    Returns:
    float: The computed sample variance.
    """
    n = len(x_hat)

    summation = 0
    for i in range(n):
        term = (x_hat[i] - x_star[i]) - (f_hat_n_x - f_hat_n_x_star)
        summation += term ** 2

    sample_variance = summation / (n - 1)
    return sample_variance


calculate confidence interval on gap

In [ ]:
def one_sided_confidence_interval(G_n_x, s_n_x_star, n, alpha=0.05):
    """
    Calculate the one-sided confidence interval:
    [0, G_n(x) + (t_(n-1,alpha) * s_n(x_star) / sqrt(n))]

    Parameters:
    G_n_x (float): The value of G_n(x_hat).
    s_n_x_star (float): The standard deviation s_n(x_star).
    n (int): The sample size.
    alpha (float): The significance level (e.g., 0.05 for 95% confidence).

    Returns:
    tuple: The confidence interval as (lower_bound, upper_bound).
    """
    if n <= 1:
        raise ValueError("Sample size must be greater than 1.")
    
    # Degrees of freedom
    df = n - 1

    # Critical t-value
    t_alpha = t.ppf(1 - alpha, df)

    # Calculate the upper bound
    upper_bound = G_n_x + (t_alpha * s_n_x_star / math.sqrt(n))

    return upper_bound


In [ ]:
G_n_x = G_n(x_hat, xi_samples)
s_n_x_star = calculate_sample_variance(x_hat, x_star, f_hat_n_x, f_hat_n_x_star)
n = len(x_hat)
gap_upper = one_sided_confidence_interval(G_n_x, s_n_x_star, n, alpha=0.05)